# Ingest constructors.json file
1. Read the file using spark dataframe reader API
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table

In [0]:
%run ../common/env_config

In [0]:
%run ../common/bronze_helpers

In [0]:
source_file = f"{landing_folder_path}/constructors.json"
table_name = f"{catalog_name}.{bronze_schema}.constructors"

#### Step 1 - Read the JSON file using the dataframe reader API

In [0]:
# Define the DDL schema
constructors_schema = """constructorId STRING, 
                         name STRING, 
                         nationality STRING, 
                         url STRING
                         """

In [0]:
# Read data from the constructors file
constructors_df = (
    spark.read
       .format('json')
       .schema(constructors_schema)
       .option('mode', 'FAILFAST')
       .load(source_file)
)

In [0]:
display(constructors_df)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
constructors_final_df = add_ingestion_metadata(constructors_df)

In [0]:
display(constructors_final_df)

#### Step 3 - Write to bronze delta table

In [0]:
(
    constructors_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))